# Building Your First AI Agent (with Ollama)

An **agent** is nothing magical. It is three things:

1. **An LLM** that can read text and write text.
2. **Tools**: ordinary Python functions the LLM is allowed to ask for.
3. **A loop**: call the LLM, run whatever tool it asks for, feed the result back, repeat until it stops asking.

```
  user question
       │
       ▼
 ┌─────────────┐   "call calculate('2+2')"   ┌──────────────┐
 │     LLM     │ ──────────────────────────► │  your Python │
 │             │ ◄────────────────────────── │   function   │
 └─────────────┘         "4"                 └──────────────┘
       │
       ▼  (no more tool calls)
   final answer
```

This notebook builds that up one piece at a time, running entirely on your Mac with Ollama.
No API keys, no cost.

**Prerequisites**: Ollama running (`ollama serve`) and a tool-capable model pulled, e.g. `ollama pull gemma4:26b`.

## 0. Setup

In [1]:
import json
import ollama

MODEL = "gemma4:26b"   # tool-capable model you have pulled. Smaller/faster: qwen2.5:7b, llama3.1 (see section 6)

# Sanity check: is the server up and the model present?
available = [m.model for m in ollama.list().models]
assert any(m.startswith(MODEL) for m in available), f"{MODEL} not found. Run: ollama pull {MODEL}"
print("Ollama is up. Using model:", MODEL)

Ollama is up. Using model: gemma4:26b


## 1. A single LLM call

Everything starts with `ollama.chat`. You pass a list of **messages**, each with a `role`
(`user`, `assistant`, `system`, or `tool`) and `content`. You get back one assistant message.

In [2]:
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is an AI agent?"}],
)
print(response.message.content)

An AI agent is an autonomous system that perceives its environment, processes information, and takes independent actions to achieve specific goals.


## 2. Memory is just a list

The model has no memory between calls. "Conversation memory" is simply you appending
each message to a list and sending the whole list every time.

In [3]:
messages = [{"role": "user", "content": "My name is Nick and I study machine learning."}]

response = ollama.chat(model=MODEL, messages=messages)
messages.append(response.message)          # remember what the assistant said
print("Assistant:", response.message.content)

messages.append({"role": "user", "content": "What is my name and what do I study?"})
response = ollama.chat(model=MODEL, messages=messages)
print("Assistant:", response.message.content)

Assistant: Nice to meet you, Nick! It’s great to meet someone diving into such a dynamic and impactful field.

Since you're studying machine learning, I can be a very useful resource for you. Feel free to use me as a tutor, a coding partner, or a sounding board. Here are a few ways I can help:

*   **Explaining Complex Concepts:** If you're struggling with backpropagation, transformers, regularization techniques, or the nuances of reinforcement learning, I can break them down into simpler terms.
*   **Coding & Implementation:** I can help you write, debug, or optimize code in **Python**, specifically using libraries like **PyTorch, TensorFlow, Scikit-learn, NumPy,** and **Pandas**.
*   **Mathematics Support:** If you need help reviewing the underlying **Linear Algebra, Calculus, or Probability and Statistics** that power ML algorithms, just ask.
*   **Paper Summarization:** If you're reading a dense research paper and want a quick summary of the architecture or the main findings, I can

Assistant: Your name is **Nick** and you study **machine learning**.


## 3. Tools are plain Python functions

A tool is a function with **type hints** and a **docstring**. Ollama reads both and turns them into a
schema the model can see. The model never runs your code. It only writes a request that says
*"please call this function with these arguments"*.

Let's define two tools.

In [4]:
from datetime import date

def calculate(expression: str) -> str:
    """Evaluate a math expression and return the result.

    Args:
        expression: A Python math expression, e.g. "2 ** 10" or "(3.5 + 1) * 4"
    """
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))   # no builtins = math only
    except Exception as e:
        return f"Error: {e}"

def get_today() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return date.today().isoformat()

print(calculate("2 ** 10"), get_today())

1024 2026-09-17


Now pass the functions to `ollama.chat` via `tools=`. Watch what comes back: **no answer**, just a
tool call request. Nothing has executed yet.

In [5]:
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "What is 17 * 23?"}],
    tools=[calculate, get_today],
)

print("Text:      ", repr(response.message.content))
print("Tool calls:", response.message.tool_calls)

Text:       ''
Tool calls: [ToolCall(function=Function(name='calculate', arguments={'expression': '17 * 23'}))]


## 4. Closing the loop by hand, once

We run the function ourselves, then send the result back as a `tool` message.
The model reads the result and writes the final answer.

In [6]:
messages = [{"role": "user", "content": "What is 17 * 23?"}]
response = ollama.chat(model=MODEL, messages=messages, tools=[calculate, get_today])
messages.append(response.message)                       # the request for a tool

call = response.message.tool_calls[0]
result = calculate(**call.function.arguments)           # WE run it
print(f"Ran {call.function.name}({call.function.arguments}) -> {result}")

messages.append({"role": "tool", "content": result, "tool_name": call.function.name})
response = ollama.chat(model=MODEL, messages=messages, tools=[calculate, get_today])
print("Final answer:", response.message.content)

Ran calculate({'expression': '17 * 23'}) -> 391


Final answer: 17 * 23 = 391


## 5. The agent loop

Step 4 was one round trip. An agent is that round trip inside a `while` loop:
keep going until the model replies **without** asking for a tool.

Two safety details worth copying into every agent you ever write:
- a **step limit**, so a confused model can't loop forever;
- **catch tool errors** and send the error text back to the model instead of crashing.

In [7]:
def run_agent(question: str, tools: list, system: str = "You are a helpful assistant. Use tools when they help.", max_steps: int = 10) -> str:
    registry = {fn.__name__: fn for fn in tools}
    messages = [{"role": "system", "content": system},
                {"role": "user",   "content": question}]

    for step in range(1, max_steps + 1):
        response = ollama.chat(model=MODEL, messages=messages, tools=tools)
        messages.append(response.message)

        if not response.message.tool_calls:            # no tool requested -> we're done
            return response.message.content

        for call in response.message.tool_calls:      # run every requested tool
            fn = registry.get(call.function.name)
            try:
                result = fn(**call.function.arguments) if fn else f"Error: unknown tool {call.function.name}"
            except Exception as e:
                result = f"Error: {e}"
            print(f"[step {step}] {call.function.name}({call.function.arguments}) -> {str(result)[:80]}")
            messages.append({"role": "tool", "content": str(result), "tool_name": call.function.name})

    return "Stopped: hit max_steps without a final answer."

In [8]:
answer = run_agent("What is 2 to the power 20, divided by 3? Give the result to 2 decimal places.", tools=[calculate, get_today])
print("\nAnswer:", answer)

[step 1] calculate({'expression': '(2**20) / 3'}) -> 349525.3333333333



Answer: 349525.33


## 6. A multi-step task

Add tools that read and write files in a `workspace/` folder. Now the model has to
**chain** several calls, where each one depends on the previous result: read, compute, write, answer.
This is where it starts to feel like an agent.

**A note on model size.** Chaining is where small models break. On this exact task, `llama3.1` (8B) requested
*all* the tools in one step and invented the file contents before reading them, so its total was wrong.
`qwen2.5:7b` got there after several failed calculator expressions. `gemma4:26b` and `qwen3:32b` did it
cleanly. The loop is identical in every case; only the model's judgment differs. Try swapping `MODEL` and watch the trace.

In [9]:
from pathlib import Path

WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

def write_file(name: str, content: str) -> str:
    """Write text to a file in the workspace folder (overwrites).

    Args:
        name: File name, e.g. "notes.txt"
        content: The text to write
    """
    (WORKSPACE / name).write_text(content)
    return f"Wrote {len(content)} characters to {name}"

def read_file(name: str) -> str:
    """Read and return the contents of a file in the workspace folder.

    Args:
        name: File name, e.g. "notes.txt"
    """
    path = WORKSPACE / name
    return path.read_text() if path.exists() else f"Error: {name} does not exist"

def list_files() -> str:
    """List the files currently in the workspace folder."""
    return ", ".join(p.name for p in WORKSPACE.iterdir()) or "(empty)"

ALL_TOOLS = [calculate, get_today, write_file, read_file, list_files]

In [10]:
# Seed a file for the agent to find
write_file("expenses.txt", "coffee 4.50\nlunch 12.25\ntrain 7.80\nbook 18.99")

task = (
    "Read expenses.txt, add up all the amounts with the calculator, "
    "and write the total to total.txt as 'Total: <amount>'. Then tell me the total."
)
answer = run_agent(task, tools=ALL_TOOLS)
print("\nAnswer:", answer)
print("total.txt contains:", read_file("total.txt"))

[step 1] list_files({}) -> expenses.txt, total.txt


[step 2] read_file({'name': 'expenses.txt'}) -> coffee 4.50
lunch 12.25
train 7.80
book 18.99


[step 3] calculate({'expression': '4.50 + 12.25 + 7.80 + 18.99'}) -> 43.54


[step 4] write_file({'content': 'Total: 43.54', 'name': 'total.txt'}) -> Wrote 12 characters to total.txt



Answer: The total amount from expenses.txt is 43.54. I have also written this value to total.txt.
total.txt contains: Total: 43.54


## 7. What you just built, and where to go next

You now have every component of a real agent:

| Piece | In this notebook |
|---|---|
| LLM call | `ollama.chat(...)` |
| Memory | the `messages` list |
| Tools | typed Python functions with docstrings |
| Loop | `run_agent` |
| Safety | `max_steps`, errors returned as text |

Production frameworks (Anthropic's tool runner, LangGraph, smolagents, ...) are this loop plus conveniences:
better prompts, retries, streaming, parallel tool calls, persistence. Once you understand the loop, they are easy to read.

**Things to try next:**
1. Change the `system` prompt in `run_agent` and see how it changes tool usage.
2. Swap `MODEL` for `qwen2.5:7b` or `llama3.1` and compare how reliably each one chains tools.
3. Add a tool that searches the web or queries a CSV with pandas.
4. Make the loop **interactive**: after each step, ask the user "run this tool? (y/n)" before executing.
5. Same loop, cloud model: Ollama's server also speaks the Anthropic Messages API, so `anthropic.Anthropic(base_url="http://localhost:11434", api_key="ollama")` runs the identical agent loop, and pointing it at the real API is a one-line change.